# NB3 — Time Travel + MERGE Upsert (lightweight)

Maps to slide §3 + deliverable bullet 3.

> Spark equivalent: `option("versionAsOf", N)` ↔ `DeltaTable(path, version=N)`
>                   `MERGE INTO ...`           ↔ `dt.merge(source, predicate)`

> **Người thực hiện:** Bùi Thị Thu Uyên · MSSV 2A202602613 · K4-Track02-Day18  
> **Đường chạy:** lightweight (`deltalake` 1.6.6, `pyiceberg` 0.12.0, DuckDB 1.5.6, Polars 1.44.2) · Python 3.11.7 · Windows 11  
> Notebook được thực thi từ đầu đến cuối; ô *Giải thích kết quả* ở cuối bài đọc các số in ở output phía trên.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path
import time
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "customers_tt")
reset(table_path)

## 1. Build version history
v0: initial 100K · v1: schema add · v2: MERGE upsert · v3: bad data

In [2]:
# v0 — initial load
v0 = pl.DataFrame({
    "customer_id": list(range(100_000)),
    "status":      ["active"] * 100_000,
    "score":       [i % 1000 for i in range(100_000)],
})
write_deltalake(table_path, v0.to_arrow(), mode="overwrite")

# v1 — add `tier` column (schema evolution)
v1 = (pl.from_arrow(DeltaTable(table_path).to_pyarrow_table())
        .with_columns(
            pl.when(pl.col("score") > 800).then(pl.lit("gold")).otherwise(pl.lit("silver")).alias("tier")
        ))
write_deltalake(table_path, v1.to_arrow(), mode="overwrite", schema_mode="overwrite")

# v2 — MERGE upsert 100K (50K updates, 50K inserts)
updates = pl.DataFrame({
    "customer_id": list(range(50_000, 150_000)),
    "status":      ["vip"] * 100_000,
    "score":       [999] * 100_000,
    "tier":        ["platinum"] * 100_000,
})
t0 = time.time()
(DeltaTable(table_path)
    .merge(source=updates.to_arrow(),
           predicate="t.customer_id = s.customer_id",
           source_alias="s", target_alias="t")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute())
print(f"MERGE 100K rows: {time.time()-t0:.2f}s")

# v3 — simulate bad data
bad = pl.DataFrame({
    "customer_id": list(range(50)),
    "status":      [None] * 50,
    "score":       [-1] * 50,
    "tier":        ["UNKNOWN"] * 50,
}, schema={"customer_id": pl.Int64, "status": pl.Utf8, "score": pl.Int64, "tier": pl.Utf8})
write_deltalake(table_path, bad.to_arrow(), mode="append")

MERGE 100K rows: 0.20s


## 2. history() — audit trail

In [3]:
for h in DeltaTable(table_path).history():
    print(f"  v{h['version']:>2}  {h['operation']:<25}  metrics={h.get('operationMetrics', {})}")

  v 3  WRITE                      metrics={'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 50, 'execution_time_ms': 2, 'num_retries': 0}
  v 2  MERGE                      metrics={'num_source_rows': 100000, 'num_target_rows_inserted': 50000, 'num_target_rows_updated': 50000, 'num_target_rows_deleted': 0, 'num_target_rows_copied': 50000, 'num_output_rows': 150000, 'num_target_files_scanned': 1, 'num_target_files_skipped_during_scan': 0, 'num_target_files_added': 1, 'num_target_files_removed': 1, 'execution_time_ms': 158, 'scan_time_ms': 112, 'rewrite_time_ms': 25}
  v 1  WRITE                      metrics={'num_added_files': 1, 'num_removed_files': 1, 'num_partitions': 0, 'num_added_rows': 100000, 'execution_time_ms': 26, 'num_retries': 0}
  v 0  WRITE                      metrics={'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 100000, 'execution_time_ms': 24, 'num_retries': 0}


## 3. Time-travel queries

In [4]:
v0_count = DeltaTable(table_path, version=0).to_pyarrow_table().num_rows
v1_cols  = DeltaTable(table_path, version=1).schema().to_arrow().names
print(f"v0 row count: {v0_count}")
print(f"v1 schema:    {v1_cols}")

v0 row count: 100000
v1 schema:    ['customer_id', 'status', 'score', 'tier']


## 4. RESTORE bad version (rollback)

`restore(2)` rewinds the *current* state of the table to whatever it was
at version 2, recorded as a new version (v4). The old versions stay in
history — restore is itself a transaction, fully auditable.

In [5]:
bad_before_restore = DeltaTable(table_path).to_pyarrow_table(filters=[("score", "<", 0)]).num_rows
print(f"Rows with score<0 at v3 (before restore): {bad_before_restore}")

t0 = time.time()
dt = DeltaTable(table_path)
dt.restore(2)
print(f"RESTORE → v2: {time.time()-t0:.2f}s   (target < 30s)")

# Verify the bad rows are gone — use delta-rs's native filter pushdown.
# (DuckDB's delta extension as of 1.5.x is stricter about post-RESTORE
# protocol entries than delta-rs writes; routing through delta-rs end-to-end
# avoids the InvalidProtocolError race.)
dt_after = DeltaTable(table_path)
bad_count = dt_after.to_pyarrow_table(filters=[("score", "<", 0)]).num_rows
print(f"Rows with score<0 after restore: {bad_count}  (expected 0)")
print(f"Current rows: {dt_after.to_pyarrow_table().num_rows:,}  (= v2 after MERGE)")

# RESTORE is a new commit, not a rewrite of history: v3 is still readable.
v3_bad = DeltaTable(table_path, version=3).to_pyarrow_table(filters=[("score", "<", 0)]).num_rows
print(f"Time travel to v3 still shows the bad rows: {v3_bad}")

Rows with score<0 at v3 (before restore): 50
RESTORE → v2: 0.05s   (target < 30s)
Rows with score<0 after restore: 0  (expected 0)
Current rows: 150,000  (= v2 after MERGE)


Time travel to v3 still shows the bad rows: 50


## 5. history() — final audit trail (now includes the RESTORE)

In [6]:
final_history = DeltaTable(table_path).history()
for h in final_history:
    print(f"  v{h['version']:>2}  {h['operation']:<25}")
print(f"\nTotal versions: {len(final_history)}  (target ≥ 5)")

  v 4  RESTORE                  
  v 3  WRITE                    
  v 2  MERGE                    
  v 1  WRITE                    
  v 0  WRITE                    

Total versions: 5  (target ≥ 5)


## 🔎 Giải thích kết quả — NB3

**Lịch sử version:** v0 WRITE (100,000 dòng) → v1 WRITE overwrite thêm cột `tier` → v2 **MERGE** → v3 WRITE append 50 dòng lỗi
→ v4 **RESTORE**. Tổng **5 version**, có cả MERGE và RESTORE ✓.

**MERGE 100K (0.20 s):** `num_source_rows = 100,000`, gồm `updated = 50,000` (id 50,000–99,999 đã tồn tại → thành `vip/999/platinum`)
và `inserted = 50,000` (id 100,000–149,999 là khách mới), kết quả `num_output_rows = 150,000`. Đây là upsert trong
**một transaction**: hoặc cả 100K thay đổi cùng được áp dụng, hoặc không thay đổi nào được áp dụng.

**Time travel:** `version=0` trả 100,000 dòng, `version=1` có schema `['customer_id', 'status', 'score', 'tier']`.
Có thể đọc lại trạng thái cũ mà không cần backup riêng.

**RESTORE:** v3 có **50** dòng `score < 0` (status null, tier `UNKNOWN`). Lệnh `restore(2)` chạy trong 0.05 s. Sau đó version
hiện tại còn **0** dòng `score < 0` và 150,000 dòng, đúng bằng trạng thái v2. RESTORE **không xóa lịch sử**: nó tạo commit mới v4,
đánh dấu remove file do v3 thêm vào. Time travel về v3 vẫn thấy 50 dòng lỗi, nên còn dấu vết để điều tra
nguồn gây lỗi. Muốn xóa hẳn file lỗi khỏi storage thì phải VACUUM sau thời gian retention (xem NB6).

## ✅ Deliverable check
- [ ] history() shows ≥ 5 versions (incl. RESTORE itself)
- [ ] MERGE 100K finished in < 60s (likely < 1s on lightweight path)
- [ ] RESTORE finished in < 30s and removed bad rows

In [7]:
ops = [h["operation"] for h in final_history]
checks = {
    "history ≥ 5 versions":          len(final_history) >= 5,
    "history includes the RESTORE":  any("RESTORE" in o.upper() for o in ops),
    "MERGE recorded in history":     any("MERGE" in o.upper() for o in ops),
    "bad rows gone after restore":   bad_count == 0,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB3 incomplete — see FAIL rows above"
print("\nNB3 complete.")

  [PASS] history ≥ 5 versions
  [PASS] history includes the RESTORE
  [PASS] MERGE recorded in history
  [PASS] bad rows gone after restore

NB3 complete.
